#**ClinRAG: Clinical Knowledge Assistant**
###**RAG-Based Medical Question Answering System**

This project aims to build a Retrieval-Augmented Generation (RAG) based Question & Answer system using medical content from the Wikipedia page on Alzheimer’s disease.
The system allows users to ask medical-related questions, retrieve relevant information from the knowledge base, and generate accurate, context-grounded answers.

###**1. Initialize the Chroma DB Connection**

In [1]:
# Initialize the embedding model

from langchain_openai import AzureOpenAIEmbeddings

keyFile = open('keys/.azure_openai_key.txt', 'r')
AZURE_OPENAI_API_KEY = keyFile.read().strip()
keyFile.close()

embeddingModel = AzureOpenAIEmbeddings(
    azure_endpoint="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    azure_deployment="text-embedding-3-small",
    api_key=AZURE_OPENAI_API_KEY)

In [2]:
# load the vector store
from langchain_chroma import Chroma

db = Chroma(collection_name="clin_rag_alzheimers", 
            embedding_function=embeddingModel,
            persist_directory="./vector_store")

###**2. Create Retriever**

In [3]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

###**3. Prompt Template**

In [4]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
You are a helpful assistant for answering questions about Alzheimer's disease.
Use the follwing context to answer the question.
Context: {context}
Answer the following question: {question}

If you don't know the answer, say you don't know.
Dont make up an answer if the answer is not in the given context.
Be concise and to the point in your answer. Dont justify your answer with the context, just give the answer.
Don't say "according to the context" or "based on the context" in your answer.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
     ]
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant for answering questions about Alzheimer\'s disease.\nUse the follwing context to answer the question.\nContext: {context}\nAnswer the following question: {question}\n\nIf you don\'t know the answer, say you don\'t know.\nDont make up an answer if the answer is not in the given context.\nBe concise and to the point in your answer. Dont justify your answer with the context, just give the answer.\nDon\'t say "according to the context" or "based on the context" in your answer.\n'), additional_kwargs={})])

###**4. Initialize the Chat Model**

In [5]:
from langchain_openai import AzureChatOpenAI

keyFile = open('keys/.azure_openai_key.txt', 'r')
AZURE_OPENAI_API_KEY = keyFile.read().strip()
keyFile.close()

chat_model = AzureChatOpenAI(
    azure_endpoint="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    azure_deployment="Training-gpt-4.1",
    api_version="2024-12-01-preview",
    api_key=AZURE_OPENAI_API_KEY)

###**5. Initialize the output Parser**

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

###**6. Create Chain**

####**6.1 Generator Chain**

In [7]:
generator_chain = prompt_template | chat_model | parser

####**6.2 RAG Chain**

In [8]:
# helper function to join the retrieved documents into a single string
def join_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [9]:
# rag chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | join_docs,
    "question": RunnablePassthrough()
} | generator_chain

###**7. Inference**

In [10]:
query = "Does genetics influence Alzheimer’s disease?"

rag_chain.invoke(query)

'Yes, genetics does influence Alzheimer’s disease.'

###**8. Additional - UI for Infernece**

In [11]:
# ! pip install ipywidgets

In [ ]:
# Input text box for question and a button to trigger the RAG chain in the notebook
import ipywidgets as widgets
from IPython.display import display, HTML

inferecne_title = widgets.HTML(value="<h1>ClinRAG: Clinical Knowledge Assistant</h1><h2>Ask a question about Alzheimer's disease</h2>")

question_input = widgets.Text(
    value='',  
    placeholder='Enter your question about Alzheimer\'s disease',
    description='Question:',
    disabled=False
)
submit_button = widgets.Button(
    description='Submit',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to submit your question',
    icon='check'
)

# Output widget with text wrapping enabled
output = widgets.Output(layout=widgets.Layout(width='100%', overflow='auto'))

def on_submit_button_clicked(b):
    with output:
        output.clear_output()
        question = question_input.value
        answer = rag_chain.invoke(question)
        # Display with HTML for proper text wrapping
        display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'><strong>Answer:</strong> {answer}</div>"))

submit_button.on_click(on_submit_button_clicked)

display(inferecne_title, question_input, submit_button, output)

HTML(value="<h1>ClinRAG: Clinical Knowledge Assistant</h1><h2>Ask a question about Alzheimer's disease</h2>")

Text(value='', description='Question:', placeholder="Enter your question about Alzheimer's disease")

Button(button_style='info', description='Submit', icon='check', style=ButtonStyle(), tooltip='Click to submit …

Output(layout=Layout(overflow='auto', width='100%'))